In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.2 MB/s eta 0:00:00


In [ ]:
import torch
import pickle
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import random
import re
import torch
from collections import Counter
import pickle

In [ ]:
device = "cuda"

# Load Judge Model

In [ ]:
model_id = "AtlaAI/Selene-1-Mini-Llama-3.1-8B"

In [ ]:
# Configuration for 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # A100/L4 GPU support bfloat16
)

In [ ]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

In [ ]:
# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
tokenizer.model_max_length = 200000
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.eos_token_id

**Change to false for similarity based examples:**

In [ ]:
use_cluster_examples = False

**Change to false for test set calculation:**

In [ ]:
use_validation = False

Load the correct files:

In [ ]:
if use_validation:
  with open("INSTANCES_100_VAL.pkl", "rb") as f:
    data = pickle.load(f)
  if use_cluster_examples:
    with open("EXAMPLES_300_VAL_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
  else:
    with open("EXAMPLES_300_VAL_SIM.pkl", "rb") as f:
      examples = pickle.load(f)
else:
  with open("INSTANCES_200_TEST.pkl", "rb") as f:
    data = pickle.load(f)
  if use_cluster_examples:
    with open("EXAMPLES_600_TEST_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
  else:
    with open("EXAMPLES_600_TEST_SIM.pkl", "rb") as f:
      examples = pickle.load(f)

In [ ]:
# Pre-group examples by instance ID (3 examples per ID)
# dictionary with keys = instance ID and value = list filled with 3 dictionaries of examples belonging to instance ID

examples_map = {}
for ex in examples:
    iid = ex['inst_id']
    if iid not in examples_map:
        examples_map[iid] = []
    examples_map[iid].append(ex)

# PHASE 1 - BASELINE ACCURACY

Define Functions:

In [ ]:
# NO EXPLANATIONS
def format_demonstrations_no_expl(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
    return demo_str.strip()

In [ ]:
# Find A, B, C, D, or E in the response
def extract_choice(response):
    #  Strict Format "**Result:** A"
    match = re.search(r"\*\*Result:\*\*\s*([A-E])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # Loose Format "Result: A"
    match = re.search(r"Result:\s*([A-E])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # if no match A-E
    return "FAIL"

Define Prompt:

In [ ]:
prompt_phase1 = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}

"""

##EVALUATION

In [ ]:
results_ph1 = {}
score_counter_ph1 = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_no_expl(current_examples)

    final_prompt = prompt_phase1.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )

    print(data_idx)
    print(final_prompt)
    print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph1[predicted_model] += 1

    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph1[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.

Target Summary:  Charlton Athletic have re-signed striker Yann Kermorgant on loan until the end of the season.

Example 2:
Input: The Real Madrid striker appeared before a judge in Versailles after being held in custody overnight.
His lawyer says he played "no part" in an alleged attempt to extort money from fellow French international Mathieu Valbuena.
The two men reportedly had an exchange during a training session.
According to French media, Mr Benzema mentioned the tape to the midfielder during a national team training session at Clairefontaine in early October.
A source close to inquiry, quoted by told AFP news agency, said the 27-year-old star told investigators on Thursday he had approached Mr Valbuena about the tape on behalf of "a childhood friend".
France coach Didier Deschamps left both players out of his squad selected on Thursday for matches against Germany and England later in November.
The prosecutor's o

Display Results:

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph1['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph1.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph1['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph1['FAIL']}")


JUDGE REPORT
Accuracy: 8.00% 

Distribution:
  Llama: 165
  Distilbart: 16
  Gold: 14
  Flan: 5


# PHASE 2 - INCLUDED EXPLANATIONS

In [ ]:
# Find A, B, C, D, or E in the response
def extract_choice(response):
    #  Strict Format "**Result:** A"
    match = re.search(r"\*\*Result:\*\*\s*([A-E])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # Loose Format "Result: A"
    match = re.search(r"Result:\s*([A-E])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    # if no match A-E
    return "FAIL"

# EXPLANATION - TOP SENTENCE

In [ ]:
def format_demonstrations_top(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Most Influential Sentence: {ex['top_sentence']['sentence']}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
    return demo_str.strip()

In [ ]:
prompt_sent_top = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Sentence: The specific sentence from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.

For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}
"""

## EVALUATION

In [ ]:
results_ph2_sent = {}
score_counter_ph2_sent = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_top(current_examples)

    final_prompt = prompt_sent_top.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )
    print(data_idx)
    print(final_prompt)
    print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph2_sent[predicted_model] += 1

    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph2_sent[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


0
 You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Sentence: The specific sentence from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.

For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoi

KeyboardInterrupt: 

Display Results:

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph2_sent['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph2_sent.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph2_sent['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph2_sent['FAIL']}")


JUDGE REPORT
Accuracy: 5.00% 

Distribution:
  Llama: 168
  Gold: 17
  Distilbart: 10
  Flan: 5


# EXPLANATION - TOP 2 SENTENCES

In [ ]:
def format_demonstrations_top2(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      top_sentences = ex.get("top_2_sentences", [])
      formatted_influential = "\n".join([f"- {s['sentence'].strip()}" for s in top_sentences])
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Most Influential Sentences: {formatted_influential}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
    return demo_str.strip()

In [ ]:
prompt_sent_top2 = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Sentences: The specific sentences from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.

For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}
"""

## EVALUATION

In [ ]:
results_ph2_sent_top2 = {}
score_counter_ph2_sent_top2 = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_top2(current_examples)

    final_prompt = prompt_sent_top2.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )

    print(data_idx)
    print(final_prompt)
    print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph2_sent_top2[predicted_model] += 1


    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph2_sent_top2[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Response E:
Labour trade union GMB has elected Peter Roache as its new general secretary.

{'A': 'gold', 'B': 'llama', 'C': 'random', 'D': 'distilbart', 'E': 'flan'}
**Reasoning:** Step 1: Infer the common style or pattern from the three examples. The Target Model tends to focus on key information, such as the outcome of a situation or the main event, while omitting less relevant details. It also tends to use simple sentence structures and concise language. Step 2: Identify which candidate matches that inferred pattern. Evaluate them.

- Response A is too vague and does not capture the essence of the election outcome.
- Response B is concise and accurately reflects the election result, including the percentage of votes and the succession of Paul Kenny.
- Response C is irrelevant to the topic of the GMB union election and does not match the Target Model's style.
- Response D contains a minor error in the name, which does

KeyboardInterrupt: 

Display Results:

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph2_sent_top2['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph2_sent_top2.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph2_sent_top2['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph2_sent_top2['FAIL']}")


JUDGE REPORT
Accuracy: 2.50% 

Distribution:
  Llama: 79
  Gold: 13
  Distilbart: 5
  Flan: 2


# EXPLANATION - TOP PHRASE (per refined sentence)

In [ ]:
def format_demonstrations_phrase(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
        input = ex["doc"]
        top_phrases = ex.get("top_phrases", {})
        sorted_phrases = sorted(
            top_phrases.values(),
            key=lambda x: x["score"],
            reverse=True
        )
        formatted_influential = ", ".join( item["phrase"].strip() for item in sorted_phrases)
        demo_str += f"Example {idx+1}:\n"
        demo_str += f"Input: {input}\n\n"
        demo_str += f"Most Influential Phrases: {formatted_influential}\n\n"
        demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"

    return demo_str.strip()

In [ ]:
prompt_phrase = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Phrases: The specific phrases from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.

For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}
"""

## EVALUATION

In [ ]:
results_ph = {}
score_counter_ph = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_phrase(current_examples)

    final_prompt = prompt_phrase.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )

    print(data_idx)
    print(final_prompt)
    print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph[predicted_model] += 1

    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Example 1:
Input: Media playback is not supported on this device
After 13 days and with 36 of the 51 games played, the traditional heavyweights are all safely through the group stage, along with all of the home nations, the Republic of Ireland - and a few so-called minnows too.
Has it been enjoyable to watch though? BBC Sport's pundits Rio Ferdinand, Thierry Henry, Alan Shearer, Danny Murphy, Mark Lawrenson and Neil Lennon give their verdict on the spectacle so far in France, which teams and players have impressed them, and how things might change in the last 16.
There are more teams here, with more to play for, but there have been fewer goals. This is the first European Championship to feature 24 teams, and 23 of them were in contention for a place in the last 16 going into the last round of group games - the exception being Ukraine. But there has been an average of 1.92 goals per game so far, only the second time it h

Display Results:

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph['FAIL']}")


JUDGE REPORT
Accuracy: 3.50% 

Distribution:
  Llama: 163
  Gold: 20
  Flan: 10
  Distilbart: 7


# EXPLANATION - TOP 2 HIGHEST SCORED PHRASES (per refined sentence)

In [ ]:
def format_demonstrations_phrase_top2(example_list):
    demo_str = ""

    for idx, ex in enumerate(example_list):
        input = ex["doc"]

        top_2_dict = ex["top_2_phrases"]

        all_phrases = [
            item["phrase"].strip()
            for phrase_list in top_2_dict.values()
            for item in phrase_list
        ]

        formatted_influential = ", ".join(all_phrases)
        demo_str += f"Example {idx+1}:\n"
        demo_str += f"Input: {input}\n\n"
        demo_str += f"Most Influential Phrases: {formatted_influential}\n\n"
        demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"

    return demo_str.strip()

In [ ]:
prompt_phrase_top2 = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–E that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Phrases: The specific phrases from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.

For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–E are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.>

**Result:** <A, B, C, D or E>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

Response E:
{Candidate5}
"""

## EVALUATION

In [ ]:
results_ph_top2 = {}
score_counter_ph_top2 = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("llama", test_instance["summaries"]["llama"]),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_phrase_top2(current_examples)

    final_prompt = prompt_phrase_top2.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
        Candidate5=candidates[4][1]
    )

    print(data_idx)
    print(final_prompt)
    print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph_top2[predicted_model] += 1

    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph_top2[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Rio Ferdinand: Dimitri Payet has been the go-to guy for France so far - everyone expected it to be Antoine Griezmann or Paul Pogba. But Payet has come out of the traps flying. He scored an unbelievable goal in the first game with his left foot and a stunning goal with his right foot when he went into the box in the second game.
We've seen it from him all season in the Premier League. But to come out in the Euros in a France shirt and do it - for me he's been the player of the tournament so far.
Alan Shearer: I have loved watching Payet. Not only his goals but his general play has been superb. I agree with Rio, I think he has been the outstanding player up to now.
Media playback is not supported on this device
Thierry Henry: Payet has not surprised me. What I like about him is that he plays like it is his last ever game. Obviously he is getting all the press and he deserves it but for me the most consistent French player

Display Results:

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph_top2['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph_top2.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph_top2['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph_top2['FAIL']}")


JUDGE REPORT
Accuracy: 5.00% 

Distribution:
  Llama: 164
  Gold: 15
  Flan: 11
  Distilbart: 10


# NO LLAMA TRIAL

In [ ]:
# Find A, B, C, D, or E in the response
def extract_choice_nollama(response):

    match = re.search(r"\*\*Result:\*\*\s*([A-D])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    match = re.search(r"Result:\s*([A-D])", response, re.IGNORECASE)
    if match: return match.group(1).upper()

    return "FAIL"

## PHASE 1 - BASELINE

In [ ]:
# NO EXPLANATIONS
def format_demonstrations_no_expl(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
    return demo_str.strip()

In [ ]:
prompt_phase1_nollama = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–D that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–D are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.>

**Result:** <A, B, C or D>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

"""

In [ ]:
results_ph1_nol = {}
score_counter_ph1_nol = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_no_expl(current_examples)

    final_prompt = prompt_phase1_nollama.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
    )

    print(data_idx)
    print(final_prompt)
    print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice_nollama(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph1_nol[predicted_model] += 1

    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph1_nol[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
62
 You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–D that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summari

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph1_nol['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph1_nol.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph1_nol['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph1_nol['FAIL']}")


JUDGE REPORT
Accuracy: 31.00% 

Distribution:
  Gold: 41
  Distilbart: 31
  Flan: 27
  Random: 1


# PHASE 2 - TOP 2 SENTENCES

In [ ]:
def format_demonstrations_top2(example_list):
    demo_str = ""
    for idx, ex in enumerate(example_list):
      input = ex["doc"]
      top_sentences = ex.get("top_2_sentences", [])
      formatted_influential = "\n".join([f"- {s['sentence'].strip()}" for s in top_sentences])
      demo_str += f"Example {idx+1}:\n"
      demo_str += f"Input: {input}\n\n"
      demo_str += f"Most Influential Sentences: {formatted_influential}\n\n"
      demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"
    return demo_str.strip()

In [ ]:
prompt_sent_top2_nollama = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–D that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Sentences: The specific sentences from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.

For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–D are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.>

**Result:** <A, B, C or D>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}

"""

In [ ]:
results_ph2_sent_top2_nol = {}
score_counter_ph2_sent_top2_nol = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_top2(current_examples)

    final_prompt = prompt_sent_top2_nollama.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
    )

    print(data_idx)
    print(final_prompt)
    print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice_nollama(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph2_sent_top2_nol[predicted_model] += 1


    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph2_sent_top2_nol[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Step 2: Identify which candidate matches that inferred pattern. Candidate D, "A trio of dancers are to perform inside Tate Britain as part of the London gallery's latest commission," aligns with the Target Model's style of focusing on key information and using simple language. It captures the essence of the original article, which is the dance performance installation at Tate Britain. This summary is concise and directly related to the main event, making it a strong match for the Target Model's behavior.

**Result:** D 

Predicted: D (gold) 

66
 You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–D that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Sentences: The specific sentences from the article that the Targ

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph2_sent_top2_nol['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph2_sent_top2_nol.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph2_sent_top2_nol['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph2_sent_top2_nol['FAIL']}")


JUDGE REPORT
Accuracy: 32.00% 

Distribution:
  Gold: 38
  Distilbart: 32
  Flan: 29
  Random: 1


## Phase 2 - Top 2 Phrases

In [ ]:
def format_demonstrations_phrase_top2(example_list):
    demo_str = ""

    for idx, ex in enumerate(example_list):
        input = ex["doc"]

        top_2_dict = ex["top_2_phrases"]

        all_phrases = [
            item["phrase"].strip()
            for phrase_list in top_2_dict.values()
            for item in phrase_list
        ]

        formatted_influential = ", ".join(all_phrases)
        demo_str += f"Example {idx+1}:\n"
        demo_str += f"Input: {input}\n\n"
        demo_str += f"Most Influential Phrases: {formatted_influential}\n\n"
        demo_str += f"Target Summary: {ex['distilbart_sum']}\n\n"

    return demo_str.strip()

In [ ]:
prompt_phrase_top2_nollama = """ You are an Expert LLM Evaluator specializing in Model Identification.
Your goal is to select the summary from candidates A–D that was produced by the Target Model.

You will be provided with 3 examples of the Target Model's behavior.
For every example you will receive:
1. The Input Article.
2. The Most Influential Phrases: The specific phrases from the article that the Target Model prioritized to create its summary.
3. The Target Model's Summary.

For a new article, your task is to select the candidate that best matches the Target Model's summarization behavior shown in the provided examples.

Here are the rules of the evaluation:
(1) You should prioritize evaluating whether the candidate summary matches the target model's style and behavior shown in the examples.
(2) You should NOT select a response because it is more fluent, detailed, or human-like. The Target Model often has specific constraints (e.g., simplistic verbs, strict length) that must be preserved.
(3) You should avoid any potential bias and your judgment should be as objective as possible. Here are some potential sources of bias:
    - The order in which the summaries are presented should NOT affect your judgment, as all candidates A–D are equally likely to be the Target Model's summary.
    - Summary length should only be a factor if it matches the Target Model's length constraints observable in the examples.

Your reply should strictly follow this format:
**Reasoning:** <Step 1: Infer the common style or pattern from the three examples. Step 2: Identify which candidate matches that inferred pattern.>

**Result:** <A, B, C or D>

Here is the data.

Target Model Demonstrations:
{demonstrations}

New Input Article:
{NewInput}

Response A:
{Candidate1}

Response B:
{Candidate2}

Response C:
{Candidate3}

Response D:
{Candidate4}
"""

In [ ]:
results_ph_top2_nol = {}
score_counter_ph_top2_nol = Counter()

for data_idx, test_instance in enumerate(data):

    inst_id = data_idx

    current_examples = examples_map.get(inst_id, [])

    if len(current_examples) < 3:
        print(f"Instance {inst_id} has fewer than 3 examples.")
        continue

    candidates = [
        ("gold", test_instance["summaries"]["gold"]),
        ("distilbart", test_instance["summaries"]["distilbart"].strip()),
        ("flan", test_instance["summaries"]["flan"]),
        ("random", test_instance["summaries"]["random"])
    ]

    random.shuffle(candidates)

    label_map = {chr(65+k): v[0] for k, v in enumerate(candidates)}

    demos = format_demonstrations_phrase_top2(current_examples)

    final_prompt = prompt_phrase_top2_nollama.format(
        demonstrations=demos,
        NewInput=test_instance["input"],
        Candidate1=candidates[0][1],
        Candidate2=candidates[1][1],
        Candidate3=candidates[2][1],
        Candidate4=candidates[3][1],
    )

    print(data_idx)
    #print(final_prompt)
    #print(label_map)

    messages = [{"role": "user", "content": final_prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(
        output_ids[0][model_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted_letter = extract_choice_nollama(response_text)
    predicted_model = label_map.get(predicted_letter, "UNKNOWN")

    score_counter_ph_top2_nol[predicted_model] += 1

    print(f"{response_text} \n")
    print(f"Predicted: {predicted_letter} ({predicted_model}) \n")

    results_ph_top2_nol[data_idx] = {
        "inst_id": inst_id,
        "model_response": response_text,
        "parsed_letter": predicted_letter,
        "predicted_model": predicted_model,
        "candidates": label_map
    }

0
**Reasoning:** To identify the Target Model's summary, we must analyze the common style or pattern from the three examples provided. Key characteristics include:

1. **Focus on key phrases**: The Target Model emphasizes specific phrases from the input article, such as "acquisitive crime" and "majority and controlling stake." This suggests that the model prioritizes concise, impactful information.
2. **Simplification of complex information**: The summaries are concise and straightforward, avoiding unnecessary details. This indicates that the model aims to distill the essence of the input article.
3. **Emphasis on key events or charges**: The summaries focus on the main charges or events, such as the trial or the plea. This suggests that the model prioritizes clarity over elaboration.

With these characteristics in mind, let's evaluate the candidate summaries:

- **A** and **D** both mention the key phrases "charity fraud" and "fraud," which aligns with the Target Model's focus on conc

In [ ]:
LIMIT = len(data)

print("JUDGE REPORT")
accuracy = (score_counter_ph_top2_nol['distilbart'] / LIMIT) * 100
print(f"Accuracy: {accuracy:.2f}% \n")
print("Distribution:")
for model_sum, count in score_counter_ph_top2_nol.most_common():
    print(f"  {model_sum.capitalize()}: {count}")

if score_counter_ph_top2_nol['FAIL'] > 0:
  print(f"\nUnparseable responses: {score_counter_ph_top2_nol['FAIL']}")


JUDGE REPORT
Accuracy: 30.00% 

Distribution:
  Gold: 44
  Distilbart: 30
  Flan: 26
